In [ ]:
!pip install --upgrade gdown
import gdown
folder_url = 'https://drive.google.com/drive/folders/1s5W0Nfz9NEN8gP14tge8GuouUDXs2Ssq'
gdown.download_folder(folder_url, quiet=False, use_cookies=False)

  Attempting uninstall: gdown
    Found existing installation: gdown 5.2.2
    Uninstalling gdown-5.2.2:
      Successfully uninstalled gdown-5.2.2


Retrieving folder contents


Processing file 1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF full-shape.zip
Processing file 1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p partial.zip
Processing file 1vwFQbYRDQlyxOZk0N0Hlxi8O_bhHxgIM rotate.zip


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF
From (redirected): https://drive.google.com/uc?id=1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF&confirm=t&uuid=dc4b14f1-1084-4da8-bc01-4b1c589d7e21
To: /content/AffordanceNet/full-shape.zip
100%|██████████| 558M/558M [00:06<00:00, 83.3MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p
From (redirected): https://drive.google.com/uc?id=1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p&confirm=t&uuid=9e7b094e-8b35-4427-b15f-5ea2fcb19095
To: /content/AffordanceNet/partial.zip
100%|██████████| 2.19G/2.19G [00:18<00:00, 116MB/s]
Downloading...
From: https://drive.google.com/uc?id=1vwFQbYRDQlyxOZk0N0Hlxi8O_bhHxgIM
To: /content/AffordanceNet/rotate.zip
100%|██████████| 1.15M/1.15M [00:00<00:00, 73.9MB/s]
Download completed


['/content/AffordanceNet/full-shape.zip',
 '/content/AffordanceNet/partial.zip',
 '/content/AffordanceNet/rotate.zip']

In [ ]:
!unzip -o /content/AffordanceNet/full-shape.zip -d /content/AffordanceNet/
!unzip -o /content/AffordanceNet/partial.zip -d /content/AffordanceNet/
!unzip -o /content/AffordanceNet/rotate.zip -d /content/AffordanceNet/
!rm /content/AffordanceNet/*.zip

Archive:  /content/AffordanceNet/full-shape.zip
  inflating: /content/AffordanceNet/full_shape_train_data.pkl  
  inflating: /content/AffordanceNet/full_shape_val_data.pkl  
Archive:  /content/AffordanceNet/partial.zip
  inflating: /content/AffordanceNet/partial_train_data.pkl  
  inflating: /content/AffordanceNet/partial_val_data.pkl  
Archive:  /content/AffordanceNet/rotate.zip
  inflating: /content/AffordanceNet/rotate_val_data.pkl  


In [ ]:
!pip install torch torchvision tqdm numpy scikit-learn

In [ ]:
# === CELL 5: Imports and Data Loading ===

import pickle
import numpy as np
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Sampler
from tqdm import tqdm
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open("/content/AffordanceNet/full_shape_train_data.pkl", "rb") as f:
    train_data = pickle.load(f)
with open("/content/AffordanceNet/full_shape_val_data.pkl", "rb") as f:
    val_data = pickle.load(f)

AFFORDANCE_NAMES = train_data[0]['affordance']

cat_affs = defaultdict(set)
for s in train_data:
    cat = s['semantic class']
    for aff in AFFORDANCE_NAMES:
        if (s['full_shape']['label'][aff].flatten() > 0.5).any():
            cat_affs[cat].add(aff)

SELECTED_CLASSES = ['grasp', 'contain', 'wrap_grasp', 'pourable']
SELECTED_INDICES = [AFFORDANCE_NAMES.index(c) for c in SELECTED_CLASSES]
NUM_CLASSES = len(SELECTED_CLASSES)
TEST_CATEGORIES = ['Mug', 'Vase', 'Scissors']
all_cats = set(s['semantic class'] for s in train_data)
TEST_CATEGORIES = [c for c in TEST_CATEGORIES if c in all_cats]

def has_positive(sample):
    for idx in SELECTED_INDICES:
        aff = AFFORDANCE_NAMES[idx]
        if (sample['full_shape']['label'][aff].flatten() > 0.5).any():
            return True
    return False

train_split = [s for s in train_data if s['semantic class'] not in TEST_CATEGORIES and has_positive(s)]
val_split   = [s for s in val_data   if s['semantic class'] not in TEST_CATEGORIES and has_positive(s)]
test_split  = [s for s in train_data if s['semantic class'] in TEST_CATEGORIES and has_positive(s)]

# Class weights
total_pos = np.zeros(NUM_CLASSES)
total_all = np.zeros(NUM_CLASSES)
for s in train_split:
    label_dict = s['full_shape']['label']
    for i, idx in enumerate(SELECTED_INDICES):
        aff = AFFORDANCE_NAMES[idx]
        mask = label_dict[aff].flatten()
        total_pos[i] += (mask > 0.5).sum()
        total_all[i] += len(mask)
total_neg = total_all - total_pos
pos_weights = np.clip(total_neg / (total_pos + 1), 1.0, 50.0)

print(f"Train: {len(train_split)}, Val: {len(val_split)}, Test: {len(test_split)}")
print(f"pos_weights: {[f'{w:.1f}' for w in pos_weights]}")

Train: 3550, Val: 498, Test: 917
pos_weights: ['38.3', '16.3', '50.0', '50.0']


In [ ]:
# === CELL 6: FIX 1 — Rich Geometric Features (13D) ===
# Instead of just normals + 1 curvature, compute 10 geometric features
# from the eigenvalue decomposition of local neighborhoods.
# These capture shape type (flat, edge, corner, sphere) much better.

def compute_rich_geometric_features(points, k_normal=30):
    """
    Input:  (N, 3) numpy float32
    Output: (N, 13) numpy float32:
        [x, y, z,                           # 3: coordinates
         nx, ny, nz,                         # 3: surface normal
         curvature,                          # 1: change of curvature
         linearity, planarity, sphericity,   # 3: shape descriptors (Demantke et al.)
         omnivariance,                       # 1: geometric complexity
         anisotropy,                         # 1: directional variation
         verticality]                        # 1: alignment with gravity (z-axis)
    """
    N = points.shape[0]
    if N < 10:
        return np.hstack([points, np.zeros((N, 10))]).astype(np.float32)

    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    pts = torch.from_numpy(points).float().to(dev)

    k = min(k_normal, N - 1)
    dists = torch.cdist(pts, pts)
    _, idx = dists.topk(k=k + 1, dim=-1, largest=False)
    idx = idx[:, 1:]

    neighbors = pts[idx]                                    # (N, k, 3)
    centered = neighbors - neighbors.mean(dim=1, keepdim=True)
    cov = torch.bmm(centered.transpose(1, 2), centered) / k
    eigvals, eigvecs = torch.linalg.eigh(cov)               # sorted ascending: λ1 ≤ λ2 ≤ λ3

    # Eigenvalues (ensure positive)
    lam1 = eigvals[:, 0].clamp(min=1e-10)  # smallest
    lam2 = eigvals[:, 1].clamp(min=1e-10)
    lam3 = eigvals[:, 2].clamp(min=1e-10)  # largest
    lam_sum = lam1 + lam2 + lam3

    # --- Surface normal (eigenvector of smallest eigenvalue) ---
    normals = eigvecs[:, :, 0]
    centroid = pts.mean(dim=0)
    outward = pts - centroid
    flip = (normals * outward).sum(dim=1) < 0
    normals[flip] = -normals[flip]

    # --- Curvature ---
    curvature = lam1 / lam_sum                               # (N,)

    # --- Shape descriptors (Demantke et al. 2011) ---
    # These are MUCH more discriminative than curvature alone!
    linearity   = (lam3 - lam2) / (lam3 + 1e-10)            # high for edges, wires
    planarity   = (lam2 - lam1) / (lam3 + 1e-10)            # high for flat surfaces
    sphericity  = lam1 / (lam3 + 1e-10)                      # high for corners, spheres

    # --- Omnivariance: geometric complexity ---
    omnivariance = (lam1 * lam2 * lam3).pow(1.0 / 3.0)

    # --- Anisotropy: how directional the local geometry is ---
    anisotropy = (lam3 - lam1) / (lam3 + 1e-10)

    # --- Verticality: alignment of normal with z-axis ---
    # Useful for distinguishing handles (horizontal) from rims (vertical)
    verticality = normals[:, 2].abs()

    result = np.hstack([
        points,                                               # (N, 3) XYZ
        normals.cpu().numpy(),                                # (N, 3) normals
        curvature.cpu().numpy()[:, None],                     # (N, 1)
        linearity.cpu().numpy()[:, None],                     # (N, 1)
        planarity.cpu().numpy()[:, None],                     # (N, 1)
        sphericity.cpu().numpy()[:, None],                    # (N, 1)
        omnivariance.cpu().numpy()[:, None],                  # (N, 1)
        anisotropy.cpu().numpy()[:, None],                    # (N, 1)
        verticality.cpu().numpy()[:, None],                   # (N, 1)
    ]).astype(np.float32)
    return result  # (N, 13)


def precompute_all(data_list, desc=""):
    augmented = []
    for s in tqdm(data_list, desc=desc):
        pts = s['full_shape']['coordinate'].astype(np.float32)
        feats = compute_rich_geometric_features(pts)
        augmented.append({
            'features': feats,
            'labels': s['full_shape']['label'],
            'category': s['semantic class']
        })
    return augmented


train_aug = precompute_all(train_split, "Train 13D features")
val_aug   = precompute_all(val_split,   "Val 13D features")
test_aug  = precompute_all(test_split,  "Test 13D features")

INPUT_DIM = train_aug[0]['features'].shape[1]
print(f"Feature dim: {INPUT_DIM}")  # Should be 13

Test 13D features: 100%|██████████| 917/917 [00:02<00:00, 354.12it/s]

Feature dim: 13


In [ ]:
# === CELL 7: Oversample (same as before) ===

class_sample_counts = np.zeros(NUM_CLASSES)
for s in train_aug:
    for i, idx in enumerate(SELECTED_INDICES):
        aff = AFFORDANCE_NAMES[idx]
        if (s['labels'][aff].flatten() > 0.5).any():
            class_sample_counts[i] += 1

max_count = class_sample_counts.max()
oversample_factors = {}
for i in range(NUM_CLASSES):
    factor = int(np.ceil(max_count / (class_sample_counts[i] + 1)))
    oversample_factors[i] = max(1, min(factor, 8))

oversampled_train = []
for s in train_aug:
    oversampled_train.append(s)
    best_factor = 1
    for i, idx in enumerate(SELECTED_INDICES):
        aff = AFFORDANCE_NAMES[idx]
        if (s['labels'][aff].flatten() > 0.5).any():
            best_factor = max(best_factor, oversample_factors[i])
    for _ in range(best_factor - 1):
        oversampled_train.append(s)

print(f"Oversampled: {len(train_aug)} → {len(oversampled_train)}")

Oversampled: 3550 → 9002


In [ ]:
# === CELL 8: FIX 3 — Dataset with Spatial Augmentation ===

class ImprovedGeoAffordanceDataset(Dataset):
    """
    Improvements over original:
    - Random rotation augmentation (SO(3)) during training
    - Random scaling (0.8-1.2)
    - Random point jittering
    - Normals are rotated consistently with points
    """
    def __init__(self, data, selected_indices, affordance_names,
                 num_points=2048, occlusion_prob=0.0, crop_angle=120,
                 augment=False):
        self.data = data
        self.selected_indices = selected_indices
        self.affordance_names = affordance_names
        self.num_points = num_points
        self.occlusion_prob = occlusion_prob
        self.crop_angle = crop_angle
        self.augment = augment

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        pts = sample['features'].copy()       # (N, 13)
        label_dict = sample['labels']

        masks = []
        for aff_idx in self.selected_indices:
            aff = self.affordance_names[aff_idx]
            masks.append((label_dict[aff].flatten() > 0.5).astype(np.float32))
        labels = np.stack(masks, axis=1)

        # ── Occlusion ──
        if np.random.random() < self.occlusion_prob:
            pts_occ, lab_occ = self._occlude(pts, labels)
            if len(pts_occ) >= 64:
                pts, labels = pts_occ, lab_occ

        # ── FIX 3: Spatial augmentation ──
        if self.augment:
            pts, labels = self._augment(pts, labels)

        # ── Normalize XYZ ──
        coords = pts[:, :3].copy()
        centroid = coords.mean(axis=0)
        coords -= centroid
        max_dist = np.max(np.linalg.norm(coords, axis=1))
        if max_dist > 1e-8:
            coords /= max_dist
        pts[:, :3] = coords

        # ── Subsample ──
        N = len(pts)
        if N >= self.num_points:
            pos_mask = labels.sum(axis=1) > 0.5
            pos_idx = np.where(pos_mask)[0]
            neg_idx = np.where(~pos_mask)[0]
            if len(pos_idx) <= self.num_points:
                n_neg = self.num_points - len(pos_idx)
                neg_chosen = np.random.choice(neg_idx, min(n_neg, len(neg_idx)), replace=False) if len(neg_idx) > 0 else np.array([], dtype=int)
                chosen = np.concatenate([pos_idx, neg_chosen])
            else:
                chosen = np.random.choice(pos_idx, self.num_points, replace=False)
            if len(chosen) < self.num_points:
                extra = np.random.choice(N, self.num_points - len(chosen), replace=True)
                chosen = np.concatenate([chosen, extra])
        else:
            chosen = np.random.choice(N, self.num_points, replace=True)

        return (torch.tensor(pts[chosen], dtype=torch.float32),
                torch.tensor(labels[chosen], dtype=torch.float32))

    def _augment(self, pts, labels):
        """Random rotation, scaling, jitter — applied CONSISTENTLY to points and normals."""
        # Random rotation around z-axis (most natural for tabletop objects)
        angle = np.random.uniform(0, 2 * np.pi)
        cos_a, sin_a = np.cos(angle), np.sin(angle)
        Rz = np.array([[cos_a, -sin_a, 0],
                        [sin_a,  cos_a, 0],
                        [0,      0,     1]], dtype=np.float32)

        # Small random tilt (up to 15 degrees)
        tilt = np.random.uniform(-0.26, 0.26)  # ~15 degrees
        cos_t, sin_t = np.cos(tilt), np.sin(tilt)
        Rx = np.array([[1, 0, 0],
                        [0, cos_t, -sin_t],
                        [0, sin_t,  cos_t]], dtype=np.float32)
        R = Rx @ Rz

        # Apply rotation to XYZ (columns 0:3)
        pts[:, :3] = pts[:, :3] @ R.T

        # Apply rotation to normals (columns 3:6)
        pts[:, 3:6] = pts[:, 3:6] @ R.T

        # Update verticality (column 12) = abs(nz) after rotation
        pts[:, 12] = np.abs(pts[:, 5])

        # Random scaling (0.85 - 1.15)
        scale = np.random.uniform(0.85, 1.15)
        pts[:, :3] *= scale

        # Random jitter on XYZ (small noise)
        jitter = np.random.normal(0, 0.005, size=pts[:, :3].shape).astype(np.float32)
        pts[:, :3] += jitter

        # Random point dropout (drop 5% of points)
        if np.random.random() < 0.3:
            keep_ratio = np.random.uniform(0.9, 0.98)
            n_keep = max(64, int(len(pts) * keep_ratio))
            keep_idx = np.random.choice(len(pts), n_keep, replace=False)
            pts = pts[keep_idx]
            labels = labels[keep_idx]

        return pts, labels

    def _occlude(self, pts, labels):
        coords = pts[:, :3]
        centroid = coords.mean(axis=0)
        centered = coords - centroid
        norms = np.linalg.norm(centered, axis=1)
        norms[norms < 1e-8] = 1e-8
        dirs = centered / norms[:, None]
        theta = np.random.uniform(0, 2 * np.pi)
        phi = np.random.uniform(0, np.pi)
        view = np.array([np.sin(phi)*np.cos(theta), np.sin(phi)*np.sin(theta), np.cos(phi)])
        visible = (dirs @ view) > np.cos(np.radians(self.crop_angle / 2))
        return pts[visible], labels[visible]

In [ ]:
# === CELL 9: FIX 2 — Multi-Scale DGCNN ===

def knn(x, k=20):
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x ** 2, dim=1, keepdim=True)
    pw = -xx - inner - xx.transpose(2, 1)
    return pw.topk(k=k, dim=-1)[1]


def get_graph_feature(x, k=20, idx=None):
    B, C, N = x.shape
    if idx is None:
        idx = knn(x, k=k)
    dev = x.device
    idx_base = torch.arange(0, B, device=dev).view(-1, 1, 1) * N
    idx_flat = (idx + idx_base).view(-1)
    x_t = x.transpose(2, 1).contiguous().view(B * N, -1)
    feat = x_t[idx_flat].view(B, N, k, C)
    x_rep = x.transpose(2, 1).view(B, N, 1, C).expand(-1, -1, k, -1)
    return torch.cat([feat - x_rep, x_rep], dim=3).permute(0, 3, 1, 2).contiguous()


class MultiScaleDGCNN(nn.Module):
    """
    Key improvements over original DGCNNGeoAff:
    1. Multi-scale KNN (k=10, 20, 40) in first layer — captures fine, medium, and coarse geometry
    2. Wider channels (64→128→256)
    3. Squeeze-and-Excitation channel attention after each layer
    4. 13D input features
    """
    def __init__(self, num_classes, input_dim=13, k_scales=[10, 20, 40]):
        super().__init__()
        self.k_scales = k_scales
        self.input_dim = input_dim

        # Multi-scale first layer: one conv per scale, then fuse
        self.conv1_branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(input_dim * 2, 64, 1, bias=False),
                nn.BatchNorm2d(64), nn.LeakyReLU(0.2))
            for _ in k_scales
        ])
        # Fuse multi-scale features
        self.fuse1 = nn.Sequential(
            nn.Conv1d(64 * len(k_scales), 128, 1, bias=False),
            nn.BatchNorm1d(128), nn.LeakyReLU(0.2))

        # Layer 2 (in learned feature space)
        self.conv2 = nn.Sequential(
            nn.Conv2d(256, 128, 1, bias=False),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2))

        # Layer 3
        self.conv3 = nn.Sequential(
            nn.Conv2d(256, 256, 1, bias=False),
            nn.BatchNorm2d(256), nn.LeakyReLU(0.2))

        # Channel attention (Squeeze-and-Excitation)
        feat_dim = 128 + 128 + 256  # = 512
        self.se = nn.Sequential(
            nn.Linear(feat_dim, feat_dim // 4),
            nn.ReLU(),
            nn.Linear(feat_dim // 4, feat_dim),
            nn.Sigmoid())

        # Head (local + global = feat_dim * 2)
        self.head = nn.Sequential(
            nn.Conv1d(feat_dim * 2, 256, 1, bias=False),
            nn.BatchNorm1d(256), nn.LeakyReLU(0.2),
            nn.Dropout(0.4),
            nn.Conv1d(256, 128, 1, bias=False),
            nn.BatchNorm1d(128), nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Conv1d(128, num_classes, 1))

    def forward(self, x):
        B, C, N = x.shape
        xyz = x[:, :3, :]

        # ── Layer 1: Multi-scale KNN on XYZ ──
        scale_feats = []
        for i, k in enumerate(self.k_scales):
            idx = knn(xyz, k=k)
            edge = get_graph_feature(x, k=k, idx=idx)
            sf = self.conv1_branches[i](edge).max(dim=-1)[0]  # (B, 64, N)
            scale_feats.append(sf)

        x1 = self.fuse1(torch.cat(scale_feats, dim=1))       # (B, 128, N)

        # ── Layer 2: learned feature space ──
        edge = get_graph_feature(x1, k=20)
        x2 = self.conv2(edge).max(dim=-1)[0]                  # (B, 128, N)

        # ── Layer 3 ──
        edge = get_graph_feature(x2, k=20)
        x3 = self.conv3(edge).max(dim=-1)[0]                  # (B, 256, N)

        # ── Multi-scale concat ──
        x_local = torch.cat([x1, x2, x3], dim=1)              # (B, 512, N)

        # ── Channel attention ──
        se_input = x_local.max(dim=-1)[0]                      # (B, 512)
        se_weight = self.se(se_input).unsqueeze(-1)            # (B, 512, 1)
        x_local = x_local * se_weight

        # ── Global feature ──
        x_global = x_local.max(dim=-1, keepdim=True)[0].expand(-1, -1, N)

        out = self.head(torch.cat([x_local, x_global], dim=1))
        return out.permute(0, 2, 1)

In [ ]:
# === CELL 10: FIX 4 — Lovász + Focal BCE Loss ===

def lovasz_grad(gt_sorted):
    """Compute gradient of the Lovász extension w.r.t sorted errors."""
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1. - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard


def lovasz_hinge_flat(logits, labels):
    """Binary Lovász hinge loss (Berman et al., CVPR 2018)."""
    if len(labels) == 0:
        return logits.sum() * 0.
    signs = 2. * labels.float() - 1.
    errors = 1. - logits * signs
    errors_sorted, perm = torch.sort(errors, dim=0, descending=True)
    perm = perm.data
    gt_sorted = labels[perm]
    grad = lovasz_grad(gt_sorted)
    loss = torch.dot(F.relu(errors_sorted), grad)
    return loss


class CombinedLoss(nn.Module):
    """
    Focal BCE + Lovász Hinge + Dice
    - Focal BCE: handles easy negatives
    - Lovász: DIRECTLY optimizes IoU (what we evaluate on!)
    - Dice: smooth overlap objective
    """
    def __init__(self, pos_weight, gamma=2.0, lovasz_weight=2.0, dice_weight=0.5):
        super().__init__()
        self.register_buffer('pos_weight', pos_weight)
        self.gamma = gamma
        self.lovasz_weight = lovasz_weight
        self.dice_weight = dice_weight

    def forward(self, pred, target):
        B, N, C = pred.shape

        # ── Focal BCE ──
        bce_raw = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        pw = self.pos_weight.unsqueeze(0).unsqueeze(0)
        weight = torch.where(target > 0.5, pw, torch.ones_like(pw))
        pt = torch.sigmoid(pred)
        pt = torch.where(target > 0.5, pt, 1 - pt)
        focal = (1 - pt) ** self.gamma
        bce = (focal * weight * bce_raw).mean()

        # ── Lovász per class ──
        lovasz = 0.0
        for c in range(C):
            lovasz += lovasz_hinge_flat(
                pred[:, :, c].reshape(-1),
                target[:, :, c].reshape(-1))
        lovasz /= C

        # ── Dice per class ──
        pred_sig = torch.sigmoid(pred)
        dice_loss = 0.0
        for c in range(C):
            inter = (pred_sig[..., c] * target[..., c]).sum()
            union = pred_sig[..., c].sum() + target[..., c].sum()
            dice_loss += 1 - (2. * inter + 1.0) / (union + 1.0)
        dice_loss /= C

        return bce + self.lovasz_weight * lovasz + self.dice_weight * dice_loss

In [ ]:
# === CELL 11: Balanced Sampler (same as before) ===

class BalancedAffordanceSampler(Sampler):
    def __init__(self, dataset, batch_size, num_classes, affordance_names, selected_indices):
        self.batch_size = batch_size
        self.num_classes = num_classes
        self.class_to_indices = defaultdict(list)
        for i, sample in enumerate(dataset.data):
            for c_idx, aff_idx in enumerate(selected_indices):
                aff = affordance_names[aff_idx]
                if (sample['labels'][aff].flatten() > 0.5).any():
                    self.class_to_indices[c_idx].append(i)
        self.num_samples = len(dataset)
        self.num_batches = self.num_samples // batch_size

    def __iter__(self):
        pools = {c: np.random.permutation(v).tolist() for c, v in self.class_to_indices.items()}
        batches = []
        for _ in range(self.num_batches):
            batch, used = [], set()
            for c in range(self.num_classes):
                if not pools[c]:
                    pools[c] = np.random.permutation(self.class_to_indices[c]).tolist()
                idx = pools[c].pop()
                if idx not in used:
                    batch.append(idx)
                    used.add(idx)
            filler = np.random.permutation(self.num_samples)
            for idx in filler:
                if len(batch) >= self.batch_size: break
                if int(idx) not in used:
                    batch.append(int(idx))
                    used.add(int(idx))
            while len(batch) < self.batch_size:
                batch.append(np.random.randint(self.num_samples))
            batches.append(batch)
        np.random.shuffle(batches)
        for b in batches: yield b

    def __len__(self): return self.num_batches

In [ ]:
# === CELL 12: Create Datasets and Loaders ===

BATCH_SIZE = 8

train_dataset = ImprovedGeoAffordanceDataset(
    oversampled_train, SELECTED_INDICES, AFFORDANCE_NAMES,
    num_points=2048, occlusion_prob=0.5, crop_angle=120,
    augment=True)  # ← FIX 3: augmentation ON

val_dataset = ImprovedGeoAffordanceDataset(
    val_aug, SELECTED_INDICES, AFFORDANCE_NAMES,
    num_points=2048, occlusion_prob=0.0, augment=False)

test_clean_ds = ImprovedGeoAffordanceDataset(
    test_aug, SELECTED_INDICES, AFFORDANCE_NAMES,
    num_points=2048, occlusion_prob=0.0, augment=False)

test_occ_ds = ImprovedGeoAffordanceDataset(
    test_aug, SELECTED_INDICES, AFFORDANCE_NAMES,
    num_points=2048, occlusion_prob=1.0, crop_angle=120, augment=False)

train_sampler = BalancedAffordanceSampler(
    train_dataset, BATCH_SIZE, NUM_CLASSES, AFFORDANCE_NAMES, SELECTED_INDICES)
train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_clean_loader = DataLoader(test_clean_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_occ_loader   = DataLoader(test_occ_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}")
print(f"Input dim: {INPUT_DIM}")

# Sanity check
pts_check, lab_check = next(iter(train_loader))
print(f"Batch shape: pts={pts_check.shape}, labels={lab_check.shape}")
for c in range(NUM_CLASSES):
    n = (lab_check[:,:,c] > 0.5).sum().item()
    print(f"  {SELECTED_CLASSES[c]}: {n} positive points")

Train batches: 1125
Input dim: 13
Batch shape: pts=torch.Size([8, 2048, 13]), labels=torch.Size([8, 2048, 4])
  grasp: 441 positive points
  contain: 1052 positive points
  wrap_grasp: 667 positive points
  pourable: 792 positive points


In [ ]:
# === CELL 13: Training Loop ===

NUM_EPOCHS = 60

model = MultiScaleDGCNN(
    num_classes=NUM_CLASSES,
    input_dim=INPUT_DIM,
    k_scales=[10, 20, 40]).to(device)

# Positive bias init
with torch.no_grad():
    model.head[-1].bias.data.fill_(1.0)

# Count parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")

# Loss
pw_tensor = torch.tensor(pos_weights, dtype=torch.float32).to(device)
criterion = CombinedLoss(pw_tensor, gamma=2.0, lovasz_weight=2.0, dice_weight=0.5).to(device)

# FIX 5: Cosine warm restart (allows model to escape local minima)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=15, T_mult=2, eta_min=1e-6)

os.makedirs("/content/output", exist_ok=True)
best_val_miou = 0.0
history = []

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for pts, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        pts = pts.permute(0, 2, 1).to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(pts)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    # Validate
    model.eval()
    iou_c = torch.zeros(NUM_CLASSES, device=device)
    cnt_c = torch.zeros(NUM_CLASSES, device=device)
    with torch.no_grad():
        for pts, labels in val_loader:
            pts = pts.permute(0,2,1).to(device)
            labels = labels.to(device)
            pred = (torch.sigmoid(model(pts)) > 0.5).float()
            for b in range(pts.size(0)):
                for c in range(NUM_CLASSES):
                    gt = labels[b,:,c]
                    if gt.sum() > 0:
                        inter = (pred[b,:,c] * gt).sum()
                        union = (pred[b,:,c] + gt).clamp(0,1).sum()
                        iou_c[c] += inter / (union + 1e-8)
                        cnt_c[c] += 1

    avg_iou = iou_c / (cnt_c + 1e-8)
    miou = avg_iou.mean().item()

    history.append({'epoch': epoch+1, 'loss': avg_loss, 'miou': miou, 'lr': current_lr})

    cls = ', '.join(f"{SELECTED_CLASSES[i]}={avg_iou[i].item():.3f}" for i in range(NUM_CLASSES))
    print(f"  Loss {avg_loss:.4f} | mIoU {miou:.4f} | lr {current_lr:.6f} | {cls}")

    if miou > best_val_miou:
        best_val_miou = miou
        torch.save(model.state_dict(), "/content/output/best_improved.pth")
        print(f"  ✅ Saved (best={best_val_miou:.4f})")

print(f"\n🏆 Best Val mIoU: {best_val_miou:.4f}")

Model parameters: 557,188


Epoch 1/60: 100%|██████████| 1125/1125 [06:17<00:00,  2.98it/s]


  Loss 3.5752 | mIoU 0.0990 | lr 0.000297 | grasp=0.068, contain=0.087, wrap_grasp=0.050, pourable=0.191
  ✅ Saved (best=0.0990)


Epoch 2/60: 100%|██████████| 1125/1125 [06:21<00:00,  2.95it/s]


  Loss 3.1630 | mIoU 0.1228 | lr 0.000287 | grasp=0.095, contain=0.116, wrap_grasp=0.058, pourable=0.222
  ✅ Saved (best=0.1228)


...

Epoch 6/60: 100%|██████████| 1125/1125 [06:20<00:00,  2.96it/s]


  Loss 3.0045 | mIoU 0.0968 | lr 0.000197 | grasp=0.101, contain=0.044, wrap_grasp=0.045, pourable=0.198


Epoch 7/60: 100%|██████████| 1125/1125 [06:20<00:00,  2.96it/s]


  Loss 2.9901 | mIoU 0.1468 | lr 0.000166 | grasp=0.104, contain=0.162, wrap_grasp=0.076, pourable=0.246
  ✅ Saved (best=0.1468)


Epoch 8/60:  70%|██████▉   | 786/1125 [04:26<01:54,  2.96it/s]

In [ ]:
# === CELL 14: Four-Condition Evaluation ===

model.load_state_dict(torch.load("/content/output/best_improved.pth", map_location=device))
model.eval()

def evaluate(loader, desc):
    iou_c = np.zeros(NUM_CLASSES)
    cnt_c = np.zeros(NUM_CLASSES)
    with torch.no_grad():
        for pts, labels in tqdm(loader, desc=desc):
            pts = pts.permute(0,2,1).to(device)
            pred = (torch.sigmoid(model(pts)) > 0.5).cpu().numpy()
            labels = labels.numpy()
            for b in range(pred.shape[0]):
                for c in range(NUM_CLASSES):
                    gt = labels[b,:,c]
                    if gt.sum() > 0:
                        inter = (pred[b,:,c] * gt).sum()
                        union = np.clip(pred[b,:,c] + gt, 0, 1).sum()
                        iou_c[c] += inter / (union + 1e-8)
                        cnt_c[c] += 1
    avg = iou_c / (cnt_c + 1e-8)
    m = avg.mean()
    print(f"\n{desc} — mIoU: {m:.4f}")
    for i, n in enumerate(SELECTED_CLASSES):
        print(f"  {n}: {avg[i]:.4f} (n={int(cnt_c[i])})")
    return m, avg

seen_clean_ds = ImprovedGeoAffordanceDataset(val_aug, SELECTED_INDICES, AFFORDANCE_NAMES,
                                              num_points=2048, occlusion_prob=0.0, augment=False)
seen_occ_ds   = ImprovedGeoAffordanceDataset(val_aug, SELECTED_INDICES, AFFORDANCE_NAMES,
                                              num_points=2048, occlusion_prob=1.0, crop_angle=120, augment=False)
seen_clean_ldr = DataLoader(seen_clean_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
seen_occ_ldr   = DataLoader(seen_occ_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("=" * 60)
print("  IMPROVED MODEL — FULL EVALUATION")
print("=" * 60)

r1, _ = evaluate(seen_clean_ldr,    "Seen / Clean")
r2, _ = evaluate(seen_occ_ldr,      "Seen / Occluded")
r3, _ = evaluate(test_clean_loader, "Unseen / Clean")
r4, _ = evaluate(test_occ_loader,   "Unseen / Occluded")

print(f"\n{'='*50}")
print(f"  COMPARISON: Old vs Improved")
print(f"{'='*50}")
print(f"{'':20s} | {'Old Clean':>10s} | {'New Clean':>10s} | {'Old Occ':>10s} | {'New Occ':>10s}")
print(f"{'-'*20}-+-{'-'*10}-+-{'-'*10}-+-{'-'*10}-+-{'-'*10}")
print(f"{'Seen':20s} | {'0.2133':>10s} | {r1:10.4f} | {'0.1829':>10s} | {r2:10.4f}")
print(f"{'Unseen':20s} | {'0.1028':>10s} | {r3:10.4f} | {'0.1041':>10s} | {r4:10.4f}")
print(f"{'='*50}")

In [ ]:
# === CELL 15: Save Training History ===

import json
with open("/content/output/history_improved.json", 'w') as f:
    json.dump(history, f)
print("History saved.")

# Plot training curves
try:
    import matplotlib.pyplot as plt
    epochs = [h['epoch'] for h in history]
    losses = [h['loss'] for h in history]
    mious = [h['miou'] for h in history]
    lrs = [h['lr'] for h in history]

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

    ax1.plot(epochs, losses, 'b-')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Training Loss')
    ax1.grid(True)

    ax2.plot(epochs, mious, 'g-')
    ax2.axhline(y=0.2132, color='r', linestyle='--', label='Old best (0.2132)')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('mIoU'); ax2.set_title('Validation mIoU')
    ax2.legend(); ax2.grid(True)

    ax3.plot(epochs, lrs, 'orange')
    ax3.set_xlabel('Epoch'); ax3.set_ylabel('LR'); ax3.set_title('Learning Rate')
    ax3.grid(True)

    plt.tight_layout()
    plt.savefig('/content/output/training_curves_improved.png', dpi=150)
    plt.show()
    print("Plot saved.")
except:
    print("Matplotlib not available, skipping plot.")